# 🧠 Hands-on: context, caching & reliability (Domain 5, 15%)

Make prompt caching actually fire (and *prove* it with the usage numbers), see a one-byte
change invalidate it, count tokens before you send, and handle the `max_tokens` truncation
stop reason. Live calls on `claude-haiku-4-5`.


In [ ]:
# Setup — run me first
%pip install -q anthropic

import os, getpass, json
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Paste your Anthropic API key (input is hidden): ")

import anthropic
client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5"   # cheap + fast for learning; swap to claude-opus-4-8 for the strongest reasoning

def check(name, fn, hint=""):
    try: ok = fn()
    except Exception as e: ok=False; hint=(hint+f"  (raised {type(e).__name__}: {e})").strip()
    print(f"{'\u2705' if ok else '\u274c'}  {name}" + (f"  \u2014 {hint}" if not ok and hint else ""))
    return ok

print("Client ready \u2713   model =", MODEL)


## A big shared context
Caching only kicks in above a minimum prefix (**4096 tokens on Haiku**), so we build a
deliberately large system prompt to cache against.


In [ ]:
BIG_CONTEXT = ("You are a support agent for Acme Cloud. Knowledge base:\n" +
    "\n".join(f"- Policy {i}: tickets are triaged within {i%24}h; refunds within 7 days; "
              f"SLA tier {i%3} covers region {i%5}." for i in range(1, 400)))
import anthropic as _a
n = client.messages.count_tokens(model=MODEL,
        system=BIG_CONTEXT, messages=[{"role": "user", "content": "hi"}]).input_tokens
print("context tokens:", n, "(need > 4096 to cache on Haiku)")


## Exercise 1 — turn caching on and prove it
Add a `cache_control` breakpoint on the system block. The **first** call writes the cache
(`cache_creation_input_tokens > 0`); the **second** reads it (`cache_read_input_tokens > 0`)
at ~10% of the price.

**TODO:** give the system block `"cache_control": {"type": "ephemeral"}`.


In [ ]:
def call(question):
    sys = [{"type": "text", "text": BIG_CONTEXT}]   # TODO: add cache_control to this block
    r = client.messages.create(model=MODEL, max_tokens=100, system=sys,
        messages=[{"role": "user", "content": question}])
    u = r.usage
    return {"write": u.cache_creation_input_tokens, "read": u.cache_read_input_tokens, "full": u.input_tokens}

first  = call("How fast are refunds?")
second = call("Which tier covers region 2?")
print("1st call:", first)
print("2nd call:", second)


In [ ]:
check("first call wrote the cache", lambda: first["write"] > 0,
      "if 0, add cache_control OR the context is under 4096 tokens")
check("second call read the cache", lambda: second["read"] > 0,
      "a cache hit means cache_read_input_tokens > 0")


## Exercise 2 — one byte invalidates the prefix
Caching is a **prefix match** — any change *before* a breakpoint invalidates everything
after it. Prepending a timestamp is the classic silent cache-killer.

**Observe:** the cache read drops back to 0.


In [ ]:
import datetime
def call_broken(question):
    # BUG on purpose: volatile text at the FRONT of the prefix
    sys = [{"type": "text", "text": f"[{datetime.datetime.now()}] " + BIG_CONTEXT,
            "cache_control": {"type": "ephemeral"}}]
    u = client.messages.create(model=MODEL, max_tokens=50, system=sys,
        messages=[{"role": "user", "content": question}]).usage
    return u.cache_read_input_tokens

a = call_broken("q1"); b = call_broken("q2")
print("cache reads with a volatile prefix:", a, b, "(both ~0 = never caching)")
check("volatile prefix defeats caching", lambda: b == 0,
      "keep timestamps/UUIDs AFTER the last cache breakpoint, never before it")


## Exercise 3 — count tokens before you send
Use `count_tokens` (never `tiktoken` — it's the wrong tokenizer for Claude) to budget
context and estimate cost up front.

**TODO:** call `client.messages.count_tokens(...)` and read `.input_tokens`.


In [ ]:
def n_tokens(text):
    # TODO: return the input_tokens for a single user message containing `text`
    return 0

small = n_tokens("Hello, Claude.")
big = n_tokens(BIG_CONTEXT)
print("small:", small, " big:", big)
check("counts are real and ordered", lambda: 0 < small < big, "use count_tokens, not len()")


## Exercise 4 — handle the max_tokens truncation
If the model hits your `max_tokens` cap it stops with `stop_reason == "max_tokens"` and the
text is **cut off**. Detect it — don't treat a truncated answer as complete.

**TODO:** set `was_truncated` by inspecting `r.stop_reason`.


In [ ]:
r = client.messages.create(model=MODEL, max_tokens=8,
    messages=[{"role": "user", "content": "Explain the CAP theorem in detail."}])
was_truncated = None   # TODO: True if the response was cut off by the token cap
print("stop_reason:", r.stop_reason, "| truncated:", was_truncated)
check("detected truncation", lambda: was_truncated is True,
      "compare r.stop_reason to 'max_tokens'")


---
**Domain-5 takeaways (exam):** caching is a prefix match — freeze the prefix, put volatile
content last, and verify with `cache_read_input_tokens`; count tokens with the API, not
`tiktoken`; and always branch on `stop_reason` (`max_tokens`, `refusal`, `tool_use`) before
trusting the content.
